Für jeden Standort:
* der wissenschaftliche Artname,
* der Namen der Aufnahme, in der die Art nachgewiesen wurde,
* die Start und Endzeit des 5-Sekundenfensters mit dem höchsten Konfidenzwert
* sowie den Konfidenzwert
für alle Arten, für die ein Konfidenzwert über >= 0,5 vorhanden ist.

Für jede nachgewiesene Art, Standort und Monitoringjahr sollte nur das Klassifikationsergebnis mit dem jeweils höchsten Konfidenzwerten gezeigt werden.

Alle Klassifikationsergebnisse für Arten, deren höchster Konfidenzwert <0,5 ist, werden für die betroffenen Standorte ignorierte.

## Connect to database
Follow the instruction in the Notebook [00_Setup_DB_Connection](./00_Setup_DB_Connection.ipynb).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Load the database username and password
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Read the username and password
username = os.getenv("DB_USERNAME")
password = os.getenv("DB_PASSWORD")

DB_CONFIG = {
    'host': 'localhost',
    'port': 8432,  # the port opened using the ssh tunnel
    'user': username,
    'password': password,
    'database': 'ecomon'
}

instance_name = "validate"
model_name = 'BNP_v3_0_glob11k_prev3_cw5s_sz2s'

In [ ]:
from bslquery import BioSoundsLabDB

# Initialize the database helper
db = BioSoundsLabDB(DB_CONFIG, verbose=True)

# Test the connection
db.test_connection()

## Estimate the size of the results table
For each site, count the number of species with confidence >= 0.5

Sum for all sites to get the row count in the final table.

In [ ]:
# List all sites and years combinations
sites = db.sites()
sites

___Caution___: This query can take a very long time.

In [ ]:
for site in sites:
    # count the number of species with confidence >= 0.5
    current_prefix = site['prefix']
    print(f"Processing site {current_prefix}")
    site['species_count'] = db.count_species_with_threshold(current_prefix, model_name, 0.5)


In [ ]:
import pandas as pd

sites_species_counts = pd.DataFrame(sites)
sites_species_counts

In [ ]:
sites_species_counts['species_count'].sum()

___Before executing the next cell, create a folder where the spreadsheets with the data will be stored.___

The folder should be called: examples/data (this folder will not be committed).

In [ ]:
# export as excel
filename = f"Site_Species_Counts_{instance_name}.xlsx".replace(' ', '_')
sites_species_counts.to_excel(
    f"data/{filename}",
    sheet_name='Data',
    index=False,
    header=True,
)

## Fetch the results

Column headers:
```
  site_prefix,site_name,
  date,time,
  filename,start_time,end_time,
  model_name,species,confidence,
```

Extra empty columns:
```
  ManualValidation,VocalizationTypeCode,
  Manual_Trigger,Manual_OtherSounds_5s,
  Notes_on_complete_snippet_
```


In [ ]:
species_above_threshold = []
for site in sites:
    print(f"Processing site {site['prefix']}")
    species_above_threshold.extend(
        db.fetch_species_with_threshold(site['prefix'], model_name, 0.5)
    )
results_df = pd.DataFrame(species_above_threshold)
results_df.head()

In [ ]:
# insert extra (empty) columns
new_columns = [
    'ManualValidation', 'VocalizationTypeCode', 'Manual_Trigger', 
    'Manual_OtherSounds_5s', 'Notes_on_complete_snippet_10_15s',
    'experimental_threshold', 'preliminary_threshold', 'final_threshold'
]
skip = results_df.shape[1]
for i, column in enumerate(new_columns):
    results_df.insert(loc=skip+i, column=column, value=None)

## Add experimental, preliminary or final thresholds

In [ ]:
unique_species = results_df['species'].unique()
for i, species_name in enumerate(unique_species):
    if i % 100 == 0:
        print("Processing %d of %d species" % (i, len(unique_species)))
    saved = db.fetch_thresholds(species_name, model_name)
    if saved is not None:
        mask = results_df['species'] == species_name
        results_df.loc[mask, 'preliminary_threshold'] = saved.get('preliminary')
        results_df.loc[mask, 'experimental_threshold'] = saved.get('experimental')
        results_df.loc[mask, 'final_threshold'] = saved.get('final')

In [ ]:
results_df.head()

## Export as Excel

In [ ]:
# export as excel
filename = f"Species_Thresholds_{instance_name}.xlsx".replace(' ', '_')
results_df.to_excel(
    f"data/{filename}",
    sheet_name='Data',
    index=False,
    header=True,
)